# IE643 - final evaluation of the student: RadGraph F1, BERTScore and safety checks (CPU)

Version `eval-final-v4.1` (2026-10-24, **all students**). Runs on **CPU** (Accelerator None, Internet On). No training. Deployed model: **K10s1_fb2** (the pre-registered v10b rule passed for both K10a_fb2 and K10s1_fb2; before that K7s1_scst). **v4 = the single final comparison of every student we trained**: all runs found in the inputs are scored (stage-1 ablations A0-A10, prefix tuning P0, SeqKD S*, SCST R*, KL-SCST K7*, no-teacher K0*, F2 K10*), with NLG metrics (BLEU-1..4, ROUGE-L, METEOR, CIDEr; same pycocoevalcap code as the distill notebook) next to CheXbert, RadGraph-XL, BERTScore, RadCliQ-v1 and the safety checks. Last cell (problem-statement extras): presence/absence agreement (accuracy + Cohen's kappa) on key findings, qualitative examples GT vs generated (`eval_examples.md/.csv`, `eval_examples.png` with the X-rays if IE643 Cleaning is attached), training curves of the deployed lineage (`eval_training_lineage.png`), checkpoint sizes. Outputs: `eval_master_table.csv/.md/.tex`, `eval_recipes.csv` (mean +- sd over seeds), `eval_all_bars.png`, `eval_scatter.png`, plus the v3 files. v3 changes vs v2: DEPLOYED, per-label table also shows K7s1_scst, extra pairs K7a -> K10a and K0s1 -> K10s1. **Attach both the v10a and the v10b commit outputs** (v10b alone lacks K7s2 / K0 / K0s1).

Why this notebook: the SCST reward and the CE14 / CE5 metrics use the same labeller (CheXbert), so part of the CheXbert gain could be Goodhart (v8 / v8.1 showed it can be gamed). RadGraph F1 uses a different model (a DyGIE++ entity-relation extractor, RadGraph-XL) and scores entities with their certainty (present / uncertain / absent) and relations, so it is the independent check.

| check | what it measures | why |
|---|---|---|
| RadGraph F1 (`simple` = entities, `partial` = entities + relation existence, `complete` = entities + exact relations; Delbrouck et al. 2022 RG_E / RG_ER / RG_ER-bar) | overlap of extracted clinical entities and relations, certainty included | independent of the CheXbert reward; **RG_ER (`partial`) is the headline** |
| BERTScore F1 (distilroberta-base, baseline-rescaled, as in CXR-Report-Metric / RadCliQ) | contextual-embedding similarity | a fluency / paraphrase check, weak on negation |
| **RadCliQ-v1** (Yu et al. 2023, reimplemented from rajpurkarlab/CXR-Report-Metric, commit 9c9ecad): RadCliQ-v1 = standardise([RadGraph F1, BERTScore, CheXbert-embedding cosine, BLEU-2]) . w + b, with the published scaler and weights (w = -0.377, -0.370, -0.253, ~0; b ~ 0). Its RadGraph term is the original RadGraph model (not XL), mean of entity F1 and relation F1 | the expected number of radiologist-flagged errors, in standardised units; **lower is better** | fitted to radiologist error counts on MIMIC-CXR, so absolute values are not comparable with papers (other dataset, impressions only, our RadGraph inference via the `radgraph` package); use it for comparisons between our systems |
| CheXbert per-label sensitivity and specificity with Wilson 95% CIs | misses vs false alarms per finding | the medical requirement; F1 alone hides which error |
| false-reassurance rate | abnormal studies (any GT finding other than Support Devices) for which the output names no finding | the clinically dangerous failure |
| hallucinated priors | share of outputs with comparison language ("interval", "stable", "unchanged", ...) when the GT has none (Ramesh et al. 2022) | one image cannot show a change; such text is unverifiable |
| copy rate, distinct-1 / distinct-2 | outputs copied verbatim from train impressions; vocabulary diversity | template collapse / memorisation |
| subgroups (if the CSV has view / sex / age) | CE14 and RG_ER per subgroup | robustness, reported only |

**Pre-registered reading (written 2026-10-24 before running this notebook):** the CheXbert gains of the v9 recipe count as **corroborated** if the paired TEST RG_ER CI of (K7s1 - A7s1) **and** of (K7a - A7) both have lower bound > 0; **contradicted** if both upper bounds < 0; otherwise **not corroborated (inconclusive)**. RadGraph does not change the adopted or deployed model (v9 rule).

**Inputs:** the output of the `klscst2` commit (all `distill_runs/*/pred_test.csv`); optionally the v8.1 commit output (distill notebook version 19, run set `scst2`: the only output holding R0r / R1r; NOT the `ie643-a10-outputs` dataset, which froze version 13 before any SCST run) and any v10 commit outputs; the log line `not available` lists any FOCUS run that no input holds; IE643 Cleaning Step 7 (`reports_clean.csv`); the teacher-cache merge version (`teacher_final/teacher_cache.jsonl`); optionally the baselines notebook output (`baseline_predictions_test.csv`).
Runtime: RadGraph-XL and the original RadGraph each take about 0.1-0.3 s per distinct text on CPU; texts are de-duplicated across systems (the students write about 100 distinct reports each), so the run takes about 25-45 min including a ~5 min environment build.

In [ ]:
import time
T0 = time.time()
NOTEBOOK_VERSION = "eval-final-v4.1"
INPUT_ROOT, WORK = "/kaggle/input", "/kaggle/working"
DEPLOYED = "K10s1_fb2"   # v10b rule passed 2026-10-24 (was K7s1_scst)
FOCUS = ["teacher", "baseline_most_common", "baseline_random_train", "A0_ce", "A0s1_ce", "A7_ce_feat", "A7s1_ce_feat", "P0_clipcap",
         "R0r_scst", "R1r_scst", "K7a_scst", "K7b_scst", "K7s1_scst", "K7s2_scst", "K0_scst", "K0s1_scst", "K10a_fb2", "K10s1_fb2"]
ALL_RUNS = True                # v4: score every run found (A1-A10, S*, R*, K*, ...); False = FOCUS list only
NLG = True                     # BLEU-1..4, ROUGE-L, METEOR (needs Java), CIDEr via pycocoevalcap (same code as the distill notebook)
# paired comparisons (a, b): b - a on TEST; skipped if either is missing
PAIRS = [("A7s1_ce_feat", "K7s1_scst"), ("A7_ce_feat", "K7a_scst"), ("A0_ce", "K7s1_scst"), ("teacher", "K7s1_scst"),
         ("R0r_scst", "K7s1_scst"), ("A0_ce", "R0r_scst"), ("K0s1_scst", "K7s1_scst"), ("K0_scst", "K7a_scst"), ("K7s1_scst", "K10s1_fb2"), ("K7a_scst", "K10a_fb2"), ("K0s1_scst", "K10s1_fb2"),
         ("A0_ce", "A7_ce_feat"), ("A0_ce", "A1_ce_kd"), ("A0_ce", "P0_clipcap"), ("A0_ce", "K0_scst"), ("teacher", "K10s1_fb2")]
# recipes reported as mean +- sd over seeds (runs that are missing are left out of the mean)
RECIPES = {"A0: CE only": ["A0_ce", "A0s1_ce", "A0s2_ce"], "A7: CE + feature KD": ["A7_ce_feat", "A7s1_ce_feat", "A7s2_ce_feat"],
           "R*r: SCST, no KL (v8.1)": ["R0r_scst", "R1r_scst"], "K7: A7 + KL-SCST (v9)": ["K7a_scst", "K7s1_scst", "K7s2_scst"],
           "K0: A0 + KL-SCST, no teacher": ["K0_scst", "K0s1_scst"], "K10: K7 + F2 reward (v10b)": ["K10a_fb2", "K10s1_fb2"]}
RG_MODEL = "radgraph-xl"
BERTSCORE = True
RADCLIQ = True                 # RadCliQ-v1: needs BERTSCORE, adds the original RadGraph model and CheXbert embeddings
# published RadCliQ-v1 (CXR-Report-Metric/CXRMetric/radcliq-v1.pkl): StandardScaler + linear weights over
# [radgraph_combined, bertscore, semb_score, bleu_score], last coefficient = intercept
RADCLIQ_V1 = {"mean": [0.53792312, 0.61757256, 0.76479421, 0.44738335], "scale": [0.30282584, 0.22430938, 0.25394391, 0.29892717],
              "coef": [-3.77083683e-01, -3.70300100e-01, -2.52616218e-01, 4.31504841e-12], "intercept": 2.46655256e-10}
N_BOOT = 1000
OFFLINE_TEST = False           # only for testing outside Kaggle (fake RadGraph / BERTScore / CheXbert)
print("notebook version:", NOTEBOOK_VERSION)

In [ ]:
import os, sys, re, json, glob, subprocess, shutil
import numpy as np, pandas as pd

KNOWN_CSV = f"{INPUT_ROOT}/notebooks/jithendraampatt/ie643-cleaning-step-7/reports_clean.csv"
csv = KNOWN_CSV if os.path.exists(KNOWN_CSV) else None
if csv is None:
    c = [p for p in glob.glob(f"{INPUT_ROOT}/**/reports_clean.csv", recursive=True) if "impression_final" in pd.read_csv(p, nrows=0).columns]
    assert len(c) == 1, f"need exactly one reports_clean.csv with impression_final, found {c}"
    csv = c[0]
df = pd.read_csv(csv, low_memory=False)
df["impression_final"] = df["impression_final"].fillna("").astype(str)
df["use_for_training"] = df["use_for_training"].astype(str).str.lower().isin(["true", "1"])
U = df[df.use_for_training]
TE, TR = U[U.subset == "test"].reset_index(drop=True), U[U.subset == "train"].reset_index(drop=True)
ORDER = TE.idx.astype(int).tolist()
GT_TEXT = TE.impression_final.tolist()
print(f"csv {csv} | test {len(TE)} | train {len(TR)}")

SYS = {}                                       # system name -> list of test outputs aligned with ORDER
tc = glob.glob(f"{INPUT_ROOT}/**/teacher_final/teacher_cache.jsonl", recursive=True)
if tc:
    TC = {json.loads(l)["idx"]: json.loads(l) for l in open(tc[0])}
    SYS["teacher"] = [str(TC[i]["impression"]).replace("**", " ").strip() for i in ORDER]   # same cleaning as the distill notebook
else:
    print("teacher_cache.jsonl not attached: no teacher row")
mc = TR.impression_final.value_counts().index[0]
SYS["baseline_most_common"] = [mc] * len(ORDER)
SYS["baseline_random_train"] = TR.impression_final.sample(len(ORDER), replace=True, random_state=0).tolist()

FOUND = {}
for p in sorted(glob.glob(f"{INPUT_ROOT}/**/distill_runs/*/pred_test.csv", recursive=True)):
    name = os.path.basename(os.path.dirname(p))
    sj = os.path.join(os.path.dirname(p), "summary.json")
    cfg = json.load(open(sj)).get("cfg", {}) if os.path.exists(sj) else {}
    if cfg.get("force_cpu") or cfg.get("max_steps_per_epoch", 0) > 0:
        continue
    if name in FOUND:
        if open(p, "rb").read() != open(FOUND[name], "rb").read():
            print(f"  WARNING {name}: different pred_test.csv in two inputs, keeping {FOUND[name]} (ignored {p})")
        continue
    FOUND[name] = p
print(f"runs found ({len(FOUND)}):", sorted(FOUND))
for name, p in FOUND.items():
    if not (ALL_RUNS or name in FOCUS):
        continue
    pr = pd.read_csv(p, keep_default_na=False).drop_duplicates("idx").set_index("idx")
    miss = len(set(ORDER) - set(pr.index))
    if miss:
        print(f"  skip {name}: {miss} test studies missing"); continue
    SYS[name] = pr.loc[ORDER, "pred"].astype(str).tolist()
for p in glob.glob(f"{INPUT_ROOT}/**/baseline_predictions_test.csv", recursive=True)[:1]:
    b = pd.read_csv(p, keep_default_na=False)
    print("baseline file", p, "columns:", list(b.columns))
    if "idx" in b.columns and set(ORDER) <= set(b.idx):
        b = b.drop_duplicates("idx").set_index("idx").loc[ORDER]
        for col in b.columns:
            if b[col].dtype == object and re.search(r"pred|nn|retriev|random|common", col, re.I) and not re.search(r"gt|ref|impression_final", col, re.I):
                SYS[f"baseline_file:{col}"] = b[col].astype(str).tolist(); print("  using baseline column", col)
missing = [f for f in FOCUS if f not in SYS]
print(f"systems scored ({len(SYS)}):", list(SYS))
print("not available (attach the commit that holds them if needed):", missing)
assert DEPLOYED in SYS, f"{DEPLOYED} predictions not found - attach the v10b commit output"

In [ ]:
RG_WORKER_PY = r"""
import json, sys, time
inp, out = sys.argv[1], sys.argv[2]
FAKE = len(sys.argv) > 3 and sys.argv[3] == "fake"
J = json.load(open(inp))
texts, pairs = J["texts"], J["pairs"]
T0 = time.time()

def _f1(h, r):
    p = sum(1 for x in h if x in r) / len(h) if h else 0.0
    q = sum(1 for x in r if x in h) / len(r) if r else 0.0
    return 2 * p * q / (p + q) if p + q > 0 else 0.0

def _cands(a, level):
    c = []
    for e in a["entities"].values():
        if level == "simple":
            c.append((e["tokens"], e["label"]))
        elif level == "partial":
            c.append((e["tokens"], e["label"], True) if e["relations"] else (e["tokens"], e["label"]))
        else:
            if not e["relations"]:
                c.append((e["tokens"], e["label"]))
            else:
                c.extend((e["tokens"].lower(), e["label"], r[0], a["entities"][r[1]]["tokens"].lower()) for r in e["relations"])
    return set(c)

def local_reward(h, r):      # same definitions as radgraph 0.1.18 rewards.py (simple, partial, complete)
    if not h["entities"] or not r["entities"]:
        return (0.0, 0.0, 0.0)
    return tuple(_f1(_cands(h, l), _cands(r, l)) for l in ("simple", "partial", "complete"))

try:
    from radgraph.rewards import compute_reward
    reward = lambda h, r: tuple(compute_reward(h, r, "all"))
    print("reward: radgraph.rewards.compute_reward", flush=True)
except ImportError:
    reward = local_reward
    print("reward: local copy", flush=True)

if FAKE:                      # offline test: entities = words, label by negation, a relation between neighbours
    def annotate(ts):
        res = []
        for t in ts:
            w = [x for x in t.lower().replace(".", " ").split() if x]
            ent = {str(k): {"tokens": x, "label": "Observation::definitely absent" if "no" in w[:k] else "Observation::definitely present",
                            "relations": [["located_at", str(k + 1)]] if k + 1 < len(w) and k % 3 == 0 else []} for k, x in enumerate(w)}
            res.append({"entities": ent})
        return res
else:
    import torch
    from radgraph import RadGraph
    rg = RadGraph(model_type=J["model"], cuda=0 if torch.cuda.is_available() else -1)
    def annotate(ts):
        d = rg([t if t.strip() else "None" for t in ts])
        return [{"entities": d[str(i)]["entities"]} for i in range(len(ts))]

def run_all(annotate, tag):
    ann = []
    CH = 100
    t0 = time.time()
    for s in range(0, len(texts), CH):
        ann.extend(annotate(texts[s:s + CH]))
        el = time.time() - t0
        print(f"RadGraph ({tag}) annotated {min(s + CH, len(texts))}/{len(texts)} texts | {el / 60:.1f} min | ~{el / min(s + CH, len(texts)) * max(len(texts) - s - CH, 0) / 60:.1f} min left", flush=True)
    return ann
ann = run_all(annotate, J["model"])
rew = [list(reward(ann[h], ann[r])) if texts[h].strip() and texts[r].strip() else [0.0, 0.0, 0.0] for h, r in pairs]
res = {"rewards": rew, "n_entities": [len(a["entities"]) for a in ann],
       "n_absent": [sum("absent" in e["label"] for e in a["entities"].values()) for a in ann]}
def _radcliq_terms():
    # RadCliQ's RadGraph term (CXR-Report-Metric radgraph_evaluate_model.py): original RadGraph model; entities {(tokens, label)},
    # relations {((tokens, label), (tokens, label), relation)}; F1 = 0 when either side is empty; combined = (entity F1 + relation F1) / 2
    def _ef(a):
        return {(e["tokens"], e["label"]) for e in a["entities"].values()}
    def _rf(a):
        E = a["entities"]
        return {((e["tokens"], e["label"]), (E[r[1]]["tokens"], E[r[1]]["label"]), r[0]) for e in E.values() for r in e["relations"] if r[1] in E}
    def _cf1(test, ret):
        tp = len(test & ret); p = tp / len(ret) if ret else 0; q = tp / len(test) if test else 0
        return 2 * p * q / (p + q) if p + q else 0.0
    if FAKE:
        ann0 = [{"entities": {k: dict(e, label=("OBS-DA" if "absent" in e["label"] else "OBS-DP")) for k, e in a["entities"].items()}} for a in ann]
    else:
        rg0 = RadGraph(model_type="radgraph", cuda=0 if torch.cuda.is_available() else -1)
        def annotate0(ts):
            d = rg0([t if t.strip() else "None" for t in ts])
            return [{"entities": d[str(i)]["entities"]} for i in range(len(ts))]
        ann0 = run_all(annotate0, "radgraph original")
    res["radgraph_combined"] = [(_cf1(_ef(ann0[r]), _ef(ann0[h])) + _cf1(_rf(ann0[r]), _rf(ann0[h]))) / 2 if texts[h].strip() else 0.0 for h, r in pairs]
    # BLEU-2 as in CXR-Report-Metric (lower-case, "." split off, whitespace tokens, weights (1/2, 1/2)); its RadCliQ-v1 weight is ~0
    import math
    from collections import Counter
    def _tok(t):
        return [w for w in str(t).lower().replace(".", " .").split(" ") if w != ""]
    def _bleu2(hyp, ref):
        h, r = _tok(hyp), _tok(ref)
        if not h:
            return 0.0
        ps = []
        for n in (1, 2):
            hc = Counter(tuple(h[i:i + n]) for i in range(len(h) - n + 1)); rc = Counter(tuple(r[i:i + n]) for i in range(len(r) - n + 1))
            tot = sum(hc.values())
            ps.append(sum(min(c, rc[g]) for g, c in hc.items()) / tot if tot else 0.0)
        if min(ps) == 0:
            return 0.0
        bp = 1.0 if len(h) > len(r) else math.exp(1 - len(r) / len(h))
        return bp * math.exp(0.5 * math.log(ps[0]) + 0.5 * math.log(ps[1]))
    res["bleu2"] = [_bleu2(texts[h], texts[r]) for h, r in pairs]
    print(f"RadCliQ RadGraph term + BLEU-2 done | {(time.time() - T0) / 60:.1f} min", flush=True)
if J.get("radcliq"):
    try:
        _radcliq_terms()
    except Exception as e:
        import traceback; traceback.print_exc()
        print(f"RadCliQ FAILED (RadGraph-XL and BERTScore unaffected): {e!r}", flush=True)
if J.get("bertscore"):
    import re as _re
    hy, rf = [_re.sub(r" +", " ", texts[h]) for h, _ in pairs], [_re.sub(r" +", " ", texts[r]) for _, r in pairs]
    if FAKE:
        res["bertscore_f1"] = [len(set(a.split()) & set(b.split())) / max(len(set(a.split()) | set(b.split())), 1) for a, b in zip(hy, rf)]
    else:
        from bert_score import BERTScorer
        sc = BERTScorer(model_type="distilroberta-base", lang="en", rescale_with_baseline=True, idf=False, batch_size=64)
        res["bertscore_f1"] = sc.score([h or "none" for h in hy], [r or "none" for r in rf])[2].tolist()
    print(f"BERTScore done: {len(pairs)} pairs | {(time.time() - T0) / 60:.1f} min", flush=True)
json.dump(res, open(out, "w"))
print("WORKER DONE", flush=True)
"""

In [ ]:
# ---- RadGraph-XL (+ BERTScore) in a private Python 3.10 env: radgraph 0.1.18 bundles allennlp-era code that is tested with transformers 4.40,
# and the Kaggle image runs Python 3.13 / transformers 5 (same approach as the teacher-cache notebook)
SRC = f"{WORK}/eval_src"; os.makedirs(SRC, exist_ok=True)
open(f"{SRC}/rg_worker.py", "w").write(RG_WORKER_PY)
ENV = "/tmp/rgenv"
PY = sys.executable
if not OFFLINE_TEST:
    PY = f"{ENV}/bin/python"
    if not os.path.exists(PY):
        t = time.time()
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
        subprocess.run(["uv", "venv", "-q", "--python", "3.10", ENV], check=True)
        r = subprocess.run(["uv", "pip", "install", "-q", "--python", PY, "torch==2.7.1", "--index-url", "https://download.pytorch.org/whl/cpu"])
        if r.returncode != 0:
            subprocess.run(["uv", "pip", "install", "-q", "--python", PY, "torch==2.7.1"], check=True)
        subprocess.run(["uv", "pip", "install", "-q", "--python", PY, "radgraph==0.1.18", "transformers==4.40.0", "bert-score==0.3.13", "nltk"], check=True)
        print(f"radgraph env built in {(time.time() - t) / 60:.1f} min")
    print(subprocess.run([PY, "-c", "import radgraph, transformers, torch; print('env ok: transformers', transformers.__version__, 'torch', torch.__version__)"],
                         capture_output=True, text=True).stdout.strip())

# de-duplicate: every distinct text is annotated once, every distinct (hypothesis, reference) pair scored once
TEXTS, TID = [], {}
def tid(t):
    t = str(t)
    if t not in TID:
        TID[t] = len(TEXTS); TEXTS.append(t)
    return TID[t]
gt_ids = [tid(t) for t in GT_TEXT]
PAIR_ID, PAIRS_L = {}, []
SYS_PAIRS = {}
for name, outs in SYS.items():
    ids = []
    for h, r in zip(outs, gt_ids):
        k = (tid(h), r)
        if k not in PAIR_ID:
            PAIR_ID[k] = len(PAIRS_L); PAIRS_L.append(k)
        ids.append(PAIR_ID[k])
    SYS_PAIRS[name] = ids
print(f"distinct texts {len(TEXTS)} (GT {len(set(GT_TEXT))}) | distinct pairs {len(PAIRS_L)} over {len(SYS)} systems")
assert not RADCLIQ or BERTSCORE, "RadCliQ needs BERTScore"
json.dump({"texts": TEXTS, "pairs": PAIRS_L, "model": RG_MODEL, "bertscore": BERTSCORE, "radcliq": RADCLIQ}, open(f"{WORK}/rg_in.json", "w"))
env = dict(os.environ, MPLBACKEND="agg", TOKENIZERS_PARALLELISM="false", PYTHONUNBUFFERED="1")
cmd = [PY, f"{SRC}/rg_worker.py", f"{WORK}/rg_in.json", f"{WORK}/rg_out.json"] + (["fake"] if OFFLINE_TEST else [])
p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, env=env)
tail = []
for line in p.stdout:
    tail = (tail + [line.rstrip()])[-40:]
    if any(k in line for k in ("annotated", "BERTScore", "RadCliQ", "DONE", "reward:", "Error", "error", "Traceback")):
        print(line.rstrip(), flush=True)
p.wait()
assert p.returncode == 0, "RadGraph worker failed:\n" + "\n".join(tail)
RGO = json.load(open(f"{WORK}/rg_out.json"))
if RADCLIQ and not ("radgraph_combined" in RGO and "bleu2" in RGO):
    print("RadCliQ terms missing (see the worker error above): RadCliQ-v1 is skipped"); RADCLIQ = False
REW = np.array(RGO["rewards"])                     # [pairs, 3] simple / partial / complete
BS = np.array(RGO["bertscore_f1"]) if BERTSCORE else None
PER = {}                                           # system -> per-study arrays
for name, ids in SYS_PAIRS.items():
    PER[name] = {"RG_E": REW[ids, 0], "RG_ER": REW[ids, 1], "RG_ER_complete": REW[ids, 2]}
    if BS is not None:
        PER[name]["BERTScore"] = BS[ids]
    if RADCLIQ:
        PER[name]["RG_combined_v1"] = np.array(RGO["radgraph_combined"])[ids]; PER[name]["BLEU2"] = np.array(RGO["bleu2"])[ids]
NENT = np.array(RGO["n_entities"])
print(f"GT impressions: mean RadGraph entities {NENT[gt_ids].mean():.1f}, with 0 entities {np.mean(NENT[gt_ids] == 0):.1%}")
print(f"sanity: RG_ER of GT vs itself would be 1; most-common-impression baseline RG_ER {PER['baseline_most_common']['RG_ER'].mean():.3f}")

In [ ]:
# ---- CheXbert labels (same labeller and convention as the distill notebook: positive = class 1, uncertain = negative)
CHEXBERT_LABELER_PY = r'''"""CheXbert labeler (Smit et al. 2020) for scoring generated report text.

Same architecture and checkpoint as the `f1chexbert` package (StanfordAIMI/RRG_scorers/chexbert.pth) but:
  * strict weight loading (only the known `position_ids` buffer may differ between transformers versions),
  * batched inference,
  * returns the raw 4-way classes so both label conventions can be computed:
      head classes 0 blank, 1 positive, 2 negative, 3 uncertain  (No Finding head: 0 blank, 1 positive)
      "pos_only"  : positive = class 1; uncertain counts as negative  (matches the IE643 baselines)
      "rrg"       : positive = class 1 or 3 (uncertain counts as positive; f1chexbert / RRG-paper convention)
"""
import numpy as np
import torch
import torch.nn as nn

CHEXBERT_ORDER = ["Enlarged Cardiomediastinum", "Cardiomegaly", "Lung Opacity", "Lung Lesion", "Edema",
                  "Consolidation", "Pneumonia", "Atelectasis", "Pneumothorax", "Pleural Effusion", "Pleural Other",
                  "Fracture", "Support Devices", "No Finding"]


class _Labeler(nn.Module):
    def __init__(self, bert):
        super().__init__()
        self.bert = bert
        h = bert.config.hidden_size
        self.linear_heads = nn.ModuleList([nn.Linear(h, 4) for _ in range(13)] + [nn.Linear(h, 2)])

    def forward(self, ids, att):
        cls = self.bert(ids, attention_mask=att)[0][:, 0]
        return [hd(cls) for hd in self.linear_heads]


class CheXbert:
    def __init__(self, device="cpu", checkpoint=None, random_init=False):
        from transformers import AutoConfig, AutoModel, BertTokenizer
        self.device = torch.device(device)
        self.tok = BertTokenizer.from_pretrained("bert-base-uncased")
        self.model = _Labeler(AutoModel.from_config(AutoConfig.from_pretrained("bert-base-uncased")))
        if not random_init:
            if checkpoint is None:
                from huggingface_hub import hf_hub_download
                checkpoint = hf_hub_download("StanfordAIMI/RRG_scorers", "chexbert.pth")
            sd = torch.load(checkpoint, map_location="cpu", weights_only=False)
            sd = sd.get("model_state_dict", sd)
            sd = {k.replace("module.", "", 1): v for k, v in sd.items()}
            res = self.model.load_state_dict(sd, strict=False)
            bad_missing = [k for k in res.missing_keys if not k.endswith("position_ids")]
            bad_unexp = [k for k in res.unexpected_keys if not k.endswith("position_ids")]
            assert not bad_missing and not bad_unexp, f"CheXbert weights mismatch: missing {bad_missing[:5]} unexpected {bad_unexp[:5]}"
        self.model.to(self.device).eval()

    @staticmethod
    def _clean(t):
        return " ".join(str(t).strip().replace("\n", " ").split())

    @torch.no_grad()
    def classes(self, texts, bs=64):
        """-> int array [N,14] of raw classes in CHEXBERT_ORDER."""
        out = []
        for s in range(0, len(texts), bs):
            chunk = [self._clean(t) for t in texts[s:s + bs]]
            # [CLS] + wordpieces + [SEP], truncated to 512 (same ids as the reference tokenize -> encode_plus path;
            # encode_plus was removed in transformers 5)
            enc = [self.tok(t, add_special_tokens=True, truncation=True, max_length=512)["input_ids"] if t else
                   [self.tok.cls_token_id, self.tok.sep_token_id] for t in chunk]
            n = max(len(e) for e in enc)
            ids = torch.zeros(len(enc), n, dtype=torch.long)
            att = torch.zeros(len(enc), n, dtype=torch.float)
            for i, e in enumerate(enc):
                ids[i, :len(e)] = torch.tensor(e); att[i, :len(e)] = 1
            logits = self.model(ids.to(self.device), att.to(self.device))
            out.append(torch.stack([l.argmax(1) for l in logits], 1).cpu())
        return torch.cat(out).numpy()

    @staticmethod
    def binary(cls, labels, convention="pos_only"):
        """raw classes [N,14] -> 0/1 matrix [N,len(labels)] in the order of `labels`."""
        J = [CHEXBERT_ORDER.index(c) for c in labels]
        c = cls[:, J]
        pos = (c == 1) if convention == "pos_only" else ((c == 1) | (c == 3))
        return pos.astype(np.int8)
'''
open(f"{SRC}/chexbert_labeler.py", "w").write(CHEXBERT_LABELER_PY)
sys.path.insert(0, SRC)
import torch
LABELS = ["No Finding", "Enlarged Cardiomediastinum", "Cardiomegaly", "Lung Opacity", "Lung Lesion", "Edema", "Consolidation",
          "Pneumonia", "Atelectasis", "Pneumothorax", "Pleural Effusion", "Pleural Other", "Fracture", "Support Devices"]
FIVE = ["Atelectasis", "Cardiomegaly", "Consolidation", "Edema", "Pleural Effusion"]
J5 = [LABELS.index(c) for c in FIVE]; NF = LABELS.index("No Finding"); SD = LABELS.index("Support Devices")
if OFFLINE_TEST:
    CLS = np.array([[(hash(t) >> k) % 4 for k in range(14)] for t in TEXTS])
    BIN = (CLS == 1).astype(bool)
else:
    from chexbert_labeler import CheXbert
    cxb = CheXbert(device="cuda" if torch.cuda.is_available() else "cpu")
    t = time.time(); CLS = cxb.classes(TEXTS); print(f"CheXbert on {len(TEXTS)} distinct texts: {(time.time() - t) / 60:.1f} min")
    BIN = CheXbert.binary(CLS, LABELS, "pos_only").astype(bool)
GTB = BIN[gt_ids]
if RADCLIQ:
    # s_emb of CXR-Report-Metric: cosine of CheXbert's final-layer [CLS] vectors (same checkpoint, cleaning and tokenisation as the labeller)
    if OFFLINE_TEST:
        EMB = np.random.RandomState(0).randn(len(TEXTS), 768).astype(np.float32)
    else:
        @torch.no_grad()
        def chexbert_cls(texts, bs=64):
            out = []
            for s in range(0, len(texts), bs):
                chunk = [CheXbert._clean(t) for t in texts[s:s + bs]]
                enc = [cxb.tok(t, add_special_tokens=True, truncation=True, max_length=512)["input_ids"] if t else
                       [cxb.tok.cls_token_id, cxb.tok.sep_token_id] for t in chunk]
                n = max(len(e) for e in enc)
                ids = torch.zeros(len(enc), n, dtype=torch.long); att = torch.zeros(len(enc), n)
                for i, e in enumerate(enc):
                    ids[i, :len(e)] = torch.tensor(e); att[i, :len(e)] = 1
                out.append(cxb.model.bert(ids.to(cxb.device), attention_mask=att.to(cxb.device))[0][:, 0].float().cpu())
            return torch.cat(out).numpy()
        t = time.time(); EMB = chexbert_cls(TEXTS); print(f"CheXbert embeddings: {(time.time() - t) / 60:.1f} min")
    EMBN = EMB / np.maximum(np.linalg.norm(EMB, axis=1, keepdims=True), 1e-12)
    _mu, _sd, _w = np.array(RADCLIQ_V1["mean"]), np.array(RADCLIQ_V1["scale"]), np.array(RADCLIQ_V1["coef"])
    for name, outs in SYS.items():
        h = np.array([tid(x) for x in outs])
        PER[name]["semb"] = (EMBN[h] * EMBN[gt_ids]).sum(1)
        X = np.stack([PER[name]["RG_combined_v1"], PER[name]["BERTScore"], PER[name]["semb"], PER[name]["BLEU2"]], 1)
        PER[name]["RadCliQ_v1"] = ((X - _mu) / _sd) @ _w + RADCLIQ_V1["intercept"]
    print("RadCliQ-v1 computed (lower = better): " + ", ".join(f"{n} {PER[n]['RadCliQ_v1'].mean():.3f}" for n in SYS))
LAB = {name: BIN[[tid(h) for h in outs]] for name, outs in SYS.items()}

def micro_f1(P, G, J=None):
    J = list(range(P.shape[1])) if J is None else J
    P, G = P[:, J], G[:, J]
    tp = (P & G).sum(); return float(2 * tp / max(2 * tp + (P & ~G).sum() + (~P & G).sum(), 1))

def wilson(k, n, z=1.96):
    if n == 0:
        return (float("nan"), float("nan"))
    p = k / n; d = 1 + z * z / n; c = (p + z * z / (2 * n)) / d; h = z * np.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / d
    return (float(c - h), float(c + h))

abnormal = GTB[:, [j for j in range(14) if j not in (NF, SD)]].any(1)      # any GT finding other than Support Devices
CLIN = {}
for name, P in LAB.items():
    named = P[:, [j for j in range(14) if j not in (NF, SD)]].any(1)
    CLIN[name] = {"CE14_F1": micro_f1(P, GTB), "CE5_F1": micro_f1(P, GTB, J5),
                  "CE14_macro_F1": float(np.mean([micro_f1(P, GTB, [j]) for j in range(14)])),
                  "CE5_macro_F1": float(np.mean([micro_f1(P, GTB, [j]) for j in J5])),
                  "CE5_precision": float((P[:, J5] & GTB[:, J5]).sum() / max(P[:, J5].sum(), 1)),
                  "CE5_recall": float((P[:, J5] & GTB[:, J5]).sum() / max(GTB[:, J5].sum(), 1)),
                  "pos_per_study": float(P[:, [j for j in range(14) if j != NF]].sum(1).mean()),
                  "CE5_specificity": float((~P[:, J5] & ~GTB[:, J5]).sum() / max((~GTB[:, J5]).sum(), 1)),
                  "CE14_F1_abnormal_only": micro_f1(P[abnormal], GTB[abnormal]),
                  "false_reassurance": float((~named[abnormal]).mean()),
                  "says_no_finding_when_abnormal": float(P[abnormal, NF].mean())}
print(f"abnormal test studies (any GT finding except Support Devices): {abnormal.sum()} of {len(abnormal)}")
# per-label sensitivity / specificity with Wilson CIs, deployed model next to the teacher and A0
rows = []
for j, c in enumerate(LABELS):
    y = GTB[:, j]; r = {"label": c, "GT_pos": int(y.sum())}
    for name in [n for n in ("teacher", "A0_ce", "A7s1_ce_feat", "K7s1_scst", "K0s1_scst", DEPLOYED) if n in LAB]:
        p = LAB[name][:, j]; tp = int((p & y).sum()); tn = int((~p & ~y).sum())
        lo, hi = wilson(tp, int(y.sum())); slo, shi = wilson(tn, int((~y).sum()))
        r[f"{name} sens"] = f"{tp / max(y.sum(), 1):.2f} [{lo:.2f}, {hi:.2f}]"
        r[f"{name} spec"] = f"{tn / max((~y).sum(), 1):.3f} [{slo:.3f}, {shi:.3f}]"
    rows.append(r)
SENS_TABLE = pd.DataFrame(rows).set_index("label")
print("\nTEST sensitivity and specificity per label (CheXbert, uncertain = negative), Wilson 95% CI:")
print(SENS_TABLE.to_string())

In [ ]:
# ---- text-level checks: hallucinated priors, copies of train impressions, diversity, length
PRIOR_RE = re.compile(r"\b(interval\w*|unchanged|stable|again|persist\w*|increas\w*|decreas\w*|improv\w*|worsen\w*|resolv\w*|redemonstrat\w*|"
                      r"re-demonstrat\w*|previous\w*|prior|compar\w*|no change|change[sd]?|remain\w*|since|new|newly|now|still|continued|progress\w*)\b")
norm = lambda t: " ".join(re.findall(r"[a-z0-9]+", str(t).lower()))
TRAIN_SET = {norm(t) for t in TR.impression_final}
gt_prior = np.array([bool(PRIOR_RE.search(norm(t))) for t in GT_TEXT])
TXT = {}
for name, outs in SYS.items():
    pr = np.array([bool(PRIOR_RE.search(norm(t))) for t in outs])
    toks = [norm(t).split() for t in outs]
    allw = [w for t in toks for w in t]; bi = [tuple(t[i:i + 2]) for t in toks for i in range(len(t) - 1)]
    TXT[name] = {"prior_ref_rate": float(pr.mean()), "hallucinated_prior": float(pr[~gt_prior].mean()) if (~gt_prior).any() else float("nan"),
                 "copy_of_train": float(np.mean([norm(t) in TRAIN_SET for t in outs])), "distinct_reports": len(set(outs)),
                 "distinct_1": len(set(allw)) / max(len(allw), 1), "distinct_2": len(set(bi)) / max(len(bi), 1),
                 "mean_words": float(np.mean([len(t) for t in toks]))}
print(f"GT: prior-reference rate {gt_prior.mean():.1%} (GT impressions are written with the prior study available; a single image is not)")

In [ ]:
# ---- NLG metrics (corpus level, same normalisation and pycocoevalcap calls as the distill notebook) + cross-check
def text_tokens(t):
    return " ".join(re.findall(r"[a-z0-9]+", str(t).lower()))

NLGR = {name: {} for name in SYS}
if NLG:
    try:
        import pycocoevalcap
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pycocoevalcap"], check=True)
    from pycocoevalcap.bleu.bleu import Bleu
    from pycocoevalcap.rouge.rouge import Rouge
    from pycocoevalcap.cider.cider import Cider
    try:
        from pycocoevalcap.meteor.meteor import Meteor
        _met = Meteor() if shutil.which("java") else None
    except Exception as e:
        _met = None; print("METEOR unavailable:", repr(e)[:150])
    if _met is None:
        print("METEOR skipped (no Java)")
    g = {i: [text_tokens(r) or "empty"] for i, r in enumerate(GT_TEXT)}
    t = time.time()
    for name, outs in SYS.items():
        p = {i: [text_tokens(x) or "empty"] for i, x in enumerate(outs)}
        try:
            b, _ = Bleu(4).compute_score(g, p, verbose=0)
        except TypeError:
            b, _ = Bleu(4).compute_score(g, p)
        r = {f"BLEU-{k + 1}": float(v) for k, v in enumerate(b)}
        r["ROUGE-L"] = float(Rouge().compute_score(g, p)[0]); r["CIDEr"] = float(Cider().compute_score(g, p)[0])
        r["METEOR"] = float(_met.compute_score(g, p)[0]) if _met is not None else float("nan")
        NLGR[name] = r
    print(f"NLG metrics for {len(SYS)} systems: {(time.time() - t) / 60:.1f} min")

# cross-check against the distill notebook's own TEST numbers (results JSON in the inputs), when present
DREF = {}
for pth in glob.glob(f"{INPUT_ROOT}/**/*.json", recursive=True):
    if os.path.getsize(pth) > 20e6 or "distill_runs" in pth:
        continue
    try:
        J = json.load(open(pth))
    except Exception:
        continue
    R = J.get("results") if isinstance(J, dict) else None
    if isinstance(R, dict) and any(str(k).endswith("|test") for k in R):
        for k, v in R.items():
            if str(k).endswith("|test") and isinstance(v, dict):
                DREF.setdefault(k.split("|")[0], v)
if DREF:
    chk = []
    for name in SYS:
        if name in DREF:
            for m, mine in (("CE14_F1", CLIN[name]["CE14_F1"]), ("CE5_F1", CLIN[name]["CE5_F1"]), ("BLEU-4", NLGR[name].get("BLEU-4")), ("ROUGE-L", NLGR[name].get("ROUGE-L"))):
                ref = DREF[name].get(m)
                if isinstance(ref, (int, float)) and mine is not None and not np.isnan(mine):
                    chk.append((name, m, abs(ref - mine)))
    if chk:
        worst = max(chk, key=lambda x: x[2])
        print(f"cross-check vs distill results JSON: {len(chk)} values, max |diff| {worst[2]:.4f} ({worst[0]} {worst[1]})"
              + ("" if worst[2] < 0.005 else "  <-- CHECK: labeller or text cleaning differs"))
else:
    print("no distill results JSON in the inputs: cross-check skipped")


In [ ]:
# ---- summary table with bootstrap CIs, paired comparisons, pre-registered reading
rs = np.random.RandomState(0)
BOOT = [rs.randint(0, len(ORDER), len(ORDER)) for _ in range(N_BOOT)]
def ci_mean(x):
    v = [x[s].mean() for s in BOOT]; return float(np.percentile(v, 2.5)), float(np.percentile(v, 97.5))
METRICS = ["RG_E", "RG_ER", "RG_ER_complete"] + (["BERTScore"] if BERTSCORE else []) + (["RadCliQ_v1", "RG_combined_v1", "semb", "BLEU2"] if RADCLIQ else [])
rows = {}
for name in SYS:
    r = {}
    for m in METRICS:
        lo, hi = ci_mean(PER[name][m]); r[m] = PER[name][m].mean(); r[f"{m}_CI"] = f"[{lo:.3f}, {hi:.3f}]"
    r.update(CLIN[name]); r.update(TXT[name]); rows[name] = r
TAB = pd.DataFrame(rows).T
order = [n for n in FOCUS if n in TAB.index] + [n for n in TAB.index if n not in FOCUS]
TAB = TAB.loc[order]
SHOW = ["RG_ER", "RG_ER_CI", "RG_E", "RG_ER_complete"] + (["BERTScore"] if BERTSCORE else []) + (["RadCliQ_v1", "RadCliQ_v1_CI", "RG_combined_v1", "semb"] if RADCLIQ else []) + ["CE14_F1", "CE5_F1", "CE5_recall", "CE5_specificity",
        "false_reassurance", "hallucinated_prior", "prior_ref_rate", "copy_of_train", "distinct_reports", "distinct_2", "mean_words"]
pd.set_option("display.width", 250)
print("\n===== TEST (", len(ORDER), "studies ) =====")
print(TAB[SHOW].to_string(float_format=lambda v: f"{v:.3f}"))

PAIRED = {}
print("\npaired bootstrap on TEST (b - a), 95% CI" + (" (RadCliQ_v1: negative = b has fewer expected errors)" if RADCLIQ else "") + ":")
for a, b in PAIRS:
    if a not in PER or b not in PER:
        continue
    PAIRED[f"{b} - {a}"] = {}
    for m in ["RG_ER", "RG_E"] + (["BERTScore"] if BERTSCORE else []) + (["RadCliQ_v1"] if RADCLIQ else []):
        d = PER[b][m] - PER[a][m]; v = [d[s].mean() for s in BOOT]
        PAIRED[f"{b} - {a}"][m] = {"mean": float(d.mean()), "ci": [float(np.percentile(v, 2.5)), float(np.percentile(v, 97.5))]}
    v = [micro_f1(LAB[b][s], GTB[s], J5) - micro_f1(LAB[a][s], GTB[s], J5) for s in BOOT]
    PAIRED[f"{b} - {a}"]["CE5_F1"] = {"mean": CLIN[b]["CE5_F1"] - CLIN[a]["CE5_F1"], "ci": [float(np.percentile(v, 2.5)), float(np.percentile(v, 97.5))]}
    print(f"  {b} - {a}: " + " | ".join(f"{m} {x['mean']:+.3f} [{x['ci'][0]:+.3f}, {x['ci'][1]:+.3f}]" for m, x in PAIRED[f"{b} - {a}"].items()))

need = ["K7s1_scst - A7s1_ce_feat", "K7a_scst - A7_ce_feat"]
READING = None
if all(k in PAIRED for k in need):
    cis = [PAIRED[k]["RG_ER"]["ci"] for k in need]
    READING = ("corroborated" if all(c[0] > 0 for c in cis) else "contradicted" if all(c[1] < 0 for c in cis) else "not corroborated (inconclusive)")
    print(f"\nPRE-REGISTERED READING: RG_ER CIs {cis} -> CheXbert gains of the v9 recipe are {READING.upper()} by RadGraph")
else:
    print("\npre-registered reading needs", need, "- missing inputs")

# subgroups (reported only): whatever view / sex / age columns the CSV has
SUB = {}
for col in [c for c in TE.columns if re.fullmatch(r"(?i)(ap_?pa|view\w*|frontal_?lateral|sex|gender|age\w*)", c)]:
    g = TE[col]
    if pd.api.types.is_numeric_dtype(g) and g.nunique() > 10:
        g = pd.cut(g, [-1, 40, 65, 200], labels=["<40", "40-65", ">65"])
    for val in g.dropna().unique():
        m = (g == val).values
        if m.sum() >= 30:
            SUB[f"{col}={val}"] = {"n": int(m.sum()), "RG_ER": float(PER[DEPLOYED]["RG_ER"][m].mean()),
                                   "CE14_F1": micro_f1(LAB[DEPLOYED][m], GTB[m]), "CE5_recall": float((LAB[DEPLOYED][m][:, J5] & GTB[m][:, J5]).sum() / max(GTB[m][:, J5].sum(), 1))}
if SUB:
    print(f"\nsubgroups for {DEPLOYED} (reported only):"); print(pd.DataFrame(SUB).T.to_string(float_format=lambda v: f"{v:.3f}"))
else:
    _cols = [c for c in TE.columns if re.fullmatch(r"(?i)(ap_?pa|view\w*|frontal_?lateral|sex|gender|age\w*)", c)]
    print(f"\nno subgroup table: candidate columns {_cols or 'none'} in reports_clean.csv (groups need >= 30 test studies)")

# examples where RadGraph and CheXbert disagree most for the deployed model (useful for the report)
cx_row = np.array([1.0 if not (LAB[DEPLOYED][i].any() or GTB[i].any()) else micro_f1(LAB[DEPLOYED][i:i + 1], GTB[i:i + 1]) for i in range(len(ORDER))])
gap = cx_row - PER[DEPLOYED]["RG_ER"]
EXAMPLES = pd.DataFrame({"idx": ORDER, "GT": GT_TEXT, DEPLOYED: SYS[DEPLOYED], "chexbert_row_F1": cx_row, "RG_ER": PER[DEPLOYED]["RG_ER"]}).iloc[np.argsort(-gap)[:8]]
print(f"\n{DEPLOYED}: largest CheXbert-row-F1 minus RG_ER gaps (label-correct but entity-poor outputs):")
for r in EXAMPLES.itertuples(index=False):
    print(f"  [{r.chexbert_row_F1:.2f} vs {r.RG_ER:.2f}] GT: {r.GT[:140]}\n                 OUT: {getattr(r, DEPLOYED)[:140]}")

TAB.to_csv(f"{WORK}/eval_final_table_test.csv")
SENS_TABLE.to_csv(f"{WORK}/eval_final_sensitivity_test.csv")
pd.DataFrame({"idx": ORDER, **{f"{n}|{m}": PER[n][m] for n in PER for m in METRICS}}).to_csv(f"{WORK}/eval_final_per_study.csv", index=False)
json.dump({"version": NOTEBOOK_VERSION, "deployed": DEPLOYED, "systems": list(SYS), "table": TAB.to_dict(orient="index"), "paired": PAIRED,
           "pre_registered_reading": READING, "subgroups": SUB, "rg_model": RG_MODEL, "bertscore": "distilroberta-base, rescaled" if BERTSCORE else None,
           "radcliq": {"version": "v1 reimplemented (CXR-Report-Metric 9c9ecad)", "params": RADCLIQ_V1, "lower_is_better": True} if RADCLIQ else None},
          open(f"{WORK}/eval_final_results.json", "w"), indent=1, default=float)

import matplotlib; matplotlib.use("agg"); import matplotlib.pyplot as plt
names = [n for n in TAB.index]
fig, ax = plt.subplots(1, 3, figsize=(17, 0.38 * len(names) + 1.5), sharey=True)
for a, m in zip(ax, ["RG_ER", "CE5_F1", "CE5_recall"]):
    vals = TAB[m].astype(float).values
    a.barh(range(len(names)), vals, color=["#c0392b" if n == DEPLOYED else "#7f8c8d" for n in names])
    if m == "RG_ER":
        lo = [ci_mean(PER[n]["RG_ER"])[0] for n in names]; hi = [ci_mean(PER[n]["RG_ER"])[1] for n in names]
        a.errorbar(vals, range(len(names)), xerr=[vals - np.array(lo), np.array(hi) - vals], fmt="none", ecolor="k", lw=1)
    a.set_title(m + (" (RadGraph, independent of the reward)" if m == "RG_ER" else " (CheXbert)")); a.grid(alpha=.3, axis="x")
ax[0].set_yticks(range(len(names))); ax[0].set_yticklabels(names); ax[0].invert_yaxis()
plt.tight_layout(); plt.savefig(f"{WORK}/eval_final.png", dpi=130)
print(f"\nsaved eval_final_results.json, eval_final_table_test.csv, eval_final_sensitivity_test.csv, eval_final_per_study.csv, eval_final.png "
      f"| total {(time.time() - T0) / 60:.1f} min\nEVAL FINAL DONE")

In [ ]:
# ---- master comparison of every student: one table, recipe means over seeds, figures
import matplotlib; matplotlib.use("agg"); import matplotlib.pyplot as plt
def group(n):
    if n == "teacher" or n.startswith("baseline"):
        return "0 reference"
    if re.match(r"A0", n):  return "1 stage 1: CE only"
    if re.match(r"A\d", n): return "2 stage 1: KD / head / prompt ablations"
    if re.match(r"P0", n):  return "3 prefix tuning (frozen LM)"
    if re.match(r"S\d", n): return "4 sequence KD (v7)"
    if re.match(r"R\dr", n): return "6 SCST, no KL (v8.1)"
    if re.match(r"R\d", n): return "5 SCST (v8)"
    if re.match(r"K0", n):  return "8 KL-SCST from A0, no teacher (v10a)"
    if re.match(r"K10", n): return "9 KL-SCST + F2 reward (v10b)"
    if re.match(r"K\d", n): return "7 KL-SCST from A7 (v9)"
    return "z other"
DESC = {"teacher": "CheXagent-2-3b (teacher)", "baseline_most_common": "most common train impression", "baseline_random_train": "random train impression",
        "A0_ce": "CE", "A1_ce_kd": "CE + token KD", "A1r_ce_kdresid": "CE + residual token KD", "A1t1_ce_kd": "CE + token KD (tau 1)",
        "A2_ce_kd_feat": "CE + token KD + feature KD", "A3_full": "all losses", "A4_ce_xattn": "CE, cross-attention connector", "A5_ce_224": "CE, 224 px",
        "A6_ce_clin": "CE + clinical head", "A7_ce_feat": "CE + feature KD", "A8_ce_clin_feat": "CE + clinical head + feature KD",
        "A9_ce_kdmask": "CE + masked token KD", "A10_ce_clin_prompt": "CE + diagnosis prompt", "P0_clipcap": "ClipCap-style prefix, frozen LM",
        "K7a_scst": "A7 -> KL-SCST beta 0.02 (pre-registered final)", "K7_scst": "A7 -> KL-SCST beta 0.02", "K7b_scst": "A7 -> KL-SCST beta 0.1", "K0_scst": "A0 -> KL-SCST, no teacher",
        "K10a_fb2": "K7a -> F2 reward", "K10s1_fb2": "K7s1 -> F2 reward", "R0r_scst": "A0 -> SCST, no KL, repetition penalty (v8.1)", "R1r_scst": "A0s1 -> SCST, no KL, repetition penalty (v8.1)",
        "R0_scst": "A0 -> SCST, no KL, no repetition penalty (v8)", "R1_scst": "A0s1 -> SCST, no KL, no repetition penalty (v8)",
        "S1_seqkd_ft": "sequence KD: teacher text ep 1-5, then GT fine-tune ep 6-8", "S1s1_seqkd_ft": "sequence KD: teacher text ep 1-5, then GT fine-tune ep 6-8 (seed 1)",
        "S2_mix": "sequence KD: target GT or teacher text (p 0.5 per epoch)", "S2s1_mix": "sequence KD: target GT or teacher text (p 0.5 per epoch) (seed 1)"}
def desc(n):
    if n in DESC: return DESC[n]
    m = re.match(r"(.+?)s(\d)(_.*)$", n)               # seed variants: A0s1_ce -> A0_ce, K7s1_scst -> K7*_scst
    base = (m.group(1) + m.group(3)) if m else None
    return (DESC.get(base, "") + f" (seed {m.group(2)})").strip() if m else ""
for n in TAB.index:
    for k, v in NLGR.get(n, {}).items():
        TAB.loc[n, k] = v
TAB.insert(0, "group", [group(n) for n in TAB.index]); TAB.insert(1, "description", [desc(n) for n in TAB.index])
TAB = TAB.loc[sorted(TAB.index, key=lambda n: (group(n), n != DEPLOYED, n))]
NLG_COLS = [c for c in ["BLEU-1", "BLEU-4", "ROUGE-L", "METEOR", "CIDEr"] if c in TAB.columns]
MASTER_COLS = (NLG_COLS + ["CE14_F1", "CE14_macro_F1", "CE5_F1", "CE5_macro_F1", "CE5_precision", "CE5_recall", "CE5_specificity", "RG_E", "RG_ER", "RG_ER_complete"]
               + (["BERTScore"] if BERTSCORE else []) + (["RadCliQ_v1"] if RADCLIQ else [])
               + ["false_reassurance", "hallucinated_prior", "copy_of_train", "pos_per_study", "distinct_reports", "mean_words"])
MASTER = TAB[["group", "description"] + MASTER_COLS].copy()
MASTER.index = [n + (" *" if n == DEPLOYED else "") for n in MASTER.index]
print(f"\n===== MASTER TABLE: {len(MASTER)} systems x {len(MASTER_COLS)} metrics, TEST {len(ORDER)} studies (* = deployed) =====")
print(MASTER.drop(columns="description").to_string(float_format=lambda v: f"{v:.3f}"))
LOWER_BETTER = {"RadCliQ_v1", "false_reassurance", "hallucinated_prior", "copy_of_train"}
students = [n for n in TAB.index if group(n)[0] not in "0"]
best = {m: (TAB.loc[students, m].astype(float).idxmin() if m in LOWER_BETTER else TAB.loc[students, m].astype(float).idxmax())
        for m in MASTER_COLS if m not in ("pos_per_study", "distinct_reports", "mean_words")}
print("\nbest student per metric:", "; ".join(f"{m}: {n}" for m, n in best.items()))

# recipes: mean +- sd over seeds
REC = {}
for rname, runs in RECIPES.items():
    have = [r for r in runs if r in TAB.index]
    if not have:
        continue
    row = {"runs": ", ".join(have), "n_seeds": len(have)}
    for m in MASTER_COLS:
        v = TAB.loc[have, m].astype(float).values
        row[m] = f"{np.nanmean(v):.3f} +- {np.nanstd(v, ddof=1):.3f}" if len(v) > 1 else f"{v[0]:.3f}"
    REC[rname] = row
RECT = pd.DataFrame(REC).T
print("\n===== RECIPES (mean +- sd over seeds) =====")
print(RECT[["n_seeds"] + [m for m in ["BLEU-4", "ROUGE-L", "CE14_F1", "CE5_F1", "CE5_macro_F1", "CE5_recall", "RG_ER", "RadCliQ_v1", "false_reassurance"] if m in RECT.columns]].to_string())

MASTER.to_csv(f"{WORK}/eval_master_table.csv"); RECT.to_csv(f"{WORK}/eval_recipes.csv")
fmt = lambda v: f"{v:.3f}" if isinstance(v, (float, np.floating)) else str(v)
with open(f"{WORK}/eval_master_table.md", "w") as f:
    f.write(f"TEST, {len(ORDER)} studies; * = deployed ({DEPLOYED}); RadCliQ-v1, false reassurance, hallucinated prior, copy: lower is better\n\n")
    try:
        f.write(MASTER.reset_index(names="system").to_markdown(index=False, floatfmt=".3f"))
    except ImportError:
        f.write(MASTER.to_string(float_format=lambda v: f"{v:.3f}"))
try:
    MASTER.drop(columns="description").to_latex(f"{WORK}/eval_master_table.tex", float_format="%.3f", longtable=True)
except Exception as e:
    print("latex export skipped:", repr(e)[:120])

GCOL = {g: c for g, c in zip(sorted({group(n) for n in TAB.index}), plt.cm.tab10.colors)}
names = list(TAB.index)
panels = [m for m in ["CE14_F1", "CE5_F1", "CE5_recall", "RG_ER", "RadCliQ_v1", "BERTScore", "ROUGE-L", "BLEU-4"] if m in TAB.columns]
fig, ax = plt.subplots(1, len(panels), figsize=(3.3 * len(panels), 0.3 * len(names) + 2), sharey=True)
for a, m in zip(ax, panels):
    vals = TAB[m].astype(float).values
    a.barh(range(len(names)), vals, color=[GCOL[group(n)] for n in names], edgecolor=["k" if n == DEPLOYED else "none" for n in names], linewidth=1.5)
    if m in PER.get(DEPLOYED, {}):
        lo = np.array([ci_mean(PER[n][m])[0] for n in names]); hi = np.array([ci_mean(PER[n][m])[1] for n in names])
        a.errorbar(vals, range(len(names)), xerr=[vals - lo, hi - vals], fmt="none", ecolor="k", lw=0.8)
    a.set_title(m + (" (lower better)" if m in LOWER_BETTER else ""), fontsize=10); a.grid(alpha=.3, axis="x")
    if m == "RadCliQ_v1":
        a.set_xlim(min(vals) - 0.1, max(vals) + 0.05)
ax[0].set_yticks(range(len(names))); ax[0].set_yticklabels([n + (" *" if n == DEPLOYED else "") for n in names], fontsize=8); ax[0].invert_yaxis()
fig.legend(handles=[plt.Rectangle((0, 0), 1, 1, color=c) for c in GCOL.values()], labels=[g[2:] for g in GCOL], loc="lower center", ncol=5, fontsize=8)
plt.tight_layout(rect=(0, 0.04, 1, 1)); plt.savefig(f"{WORK}/eval_all_bars.png", dpi=130); plt.close()

ys = [m for m in ["RG_ER", "RadCliQ_v1"] if m in TAB.columns]
fig, ax = plt.subplots(1, len(ys), figsize=(7 * len(ys), 6))
for a, m in zip(np.atleast_1d(ax), ys):
    for n in names:
        x, y = float(TAB.loc[n, "CE5_F1"]), float(TAB.loc[n, m])
        a.scatter(x, y, color=GCOL[group(n)], s=70 if n == DEPLOYED else 30, edgecolor="k" if n == DEPLOYED else "none")
        a.annotate(n.replace("_scst", "").replace("_ce", "").replace("_feat", "f"), (x, y), fontsize=7, xytext=(3, 2), textcoords="offset points")
    a.set_xlabel("CE5 micro-F1 (CheXbert, = the SCST reward's labeller)"); a.set_ylabel(m + (" (lower better)" if m in LOWER_BETTER else " (independent of the reward)"))
    a.grid(alpha=.3)
plt.suptitle("Clinical label accuracy vs independent text metrics: up-right (RG_ER) / down-right (RadCliQ) is better; reward hacking = right but bad on y")
plt.tight_layout(); plt.savefig(f"{WORK}/eval_scatter.png", dpi=130); plt.close()
json.dump({"version": NOTEBOOK_VERSION, "deployed": DEPLOYED, "master": TAB.to_dict(orient="index"), "recipes": REC, "best_per_metric": best,
           "nlg_cross_check_runs": sorted(DREF)}, open(f"{WORK}/eval_master.json", "w"), indent=1, default=float)
print(f"saved eval_master_table.csv/.md/.tex, eval_recipes.csv, eval_master.json, eval_all_bars.png, eval_scatter.png | total {(time.time() - T0) / 60:.1f} min\nEVAL MASTER DONE")


In [ ]:
# ---- report extras required by the problem statement: presence/absence agreement on key findings, qualitative examples
#      (generated vs ground truth, with the X-ray when IE643 Cleaning is attached), training curves of the deployed model's lineage, model size
import textwrap
KEY = ["No Finding", "Atelectasis", "Cardiomegaly", "Consolidation", "Edema", "Pleural Effusion", "Pneumothorax", "Pneumonia", "Lung Opacity", "Support Devices"]
def kappa(p, y):
    po = float((p == y).mean()); pe = float(p.mean() * y.mean() + (1 - p.mean()) * (1 - y.mean()))
    return (po - pe) / (1 - pe) if pe < 1 else float("nan")
AGREE_SYS = [n for n in ("teacher", "A0_ce", "A7s1_ce_feat", "K7s1_scst", DEPLOYED) if n in LAB]
rows = []
for c in KEY:
    j = LABELS.index(c); y = GTB[:, j]; r = {"finding": c, "GT_present": int(y.sum())}
    for n in AGREE_SYS:
        p = LAB[n][:, j]
        r[f"{n} agree"] = float((p == y).mean()); r[f"{n} kappa"] = kappa(p, y)
    rows.append(r)
AGREE = pd.DataFrame(rows).set_index("finding")
print("\nPRESENCE / ABSENCE AGREEMENT on key findings, TEST (CheXbert labels of output vs of GT; agree = accuracy, kappa = Cohen's kappa):")
print(AGREE.to_string(float_format=lambda v: f"{v:.3f}"))
AGREE.to_csv(f"{WORK}/eval_key_finding_agreement.csv")

# model size (fp16 checkpoint on disk) per run
for n in TAB.index:
    if n in FOUND:
        ck = os.path.join(os.path.dirname(FOUND[n]), "best.pt")
        TAB.loc[n, "ckpt_MB"] = os.path.getsize(ck) / 2**20 if os.path.exists(ck) else float("nan")
if DEPLOYED in FOUND:
    print(f"\n{DEPLOYED} checkpoint: {TAB.loc[DEPLOYED, 'ckpt_MB']:.0f} MB on disk (includes train-only heads); teacher CheXagent-2-3b ~3B parameters")

# qualitative examples: one hit per key finding, the two failure types, and random studies (fixed seed)
rs = np.random.RandomState(7)
D = LAB[DEPLOYED]; named = D[:, [j for j in range(14) if j not in (NF, SD)]].any(1)
pick, why = [], []
def add(cands, reason):
    cands = [i for i in cands if i not in pick]
    if cands:
        pick.append(int(rs.choice(cands))); why.append(reason)
add(np.where(~abnormal & D[:, NF])[0], "normal, correctly normal")
for c in ["Pleural Effusion", "Edema", "Cardiomegaly", "Atelectasis", "Consolidation", "Pneumothorax"]:
    j = LABELS.index(c); add(np.where(GTB[:, j] & D[:, j])[0], f"{c}: found")
add(np.where(abnormal & ~named)[0], "FAILURE: false reassurance (abnormal, nothing named)")
add(np.where(~abnormal & named)[0], "FAILURE: finding named in a normal study")
for _ in range(3):
    add(list(range(len(ORDER))), "random")
lab_str = lambda row: ", ".join(LABELS[j] for j in np.where(row)[0]) or "-"
EXS = []
for i, w in zip(pick, why):
    e = {"idx": ORDER[i], "why": w, "GT": GT_TEXT[i], "GT_labels": lab_str(GTB[i])}
    for n in [x for x in ("teacher", "A0_ce", "K7s1_scst", DEPLOYED) if x in SYS]:
        e[n] = SYS[n][i]; e[f"{n}_labels"] = lab_str(LAB[n][i])
    e[f"{DEPLOYED}_RG_ER"] = float(PER[DEPLOYED]["RG_ER"][i])
    EXS.append(e)
EXDF = pd.DataFrame(EXS); EXDF.to_csv(f"{WORK}/eval_examples.csv", index=False)
with open(f"{WORK}/eval_examples.md", "w") as f:
    for e in EXS:
        f.write(f"### study {e['idx']} ({e['why']})\n\n| source | report | CheXbert labels |\n|---|---|---|\n| **ground truth** | {e['GT']} | {e['GT_labels']} |\n")
        for n in [x for x in ("teacher", "A0_ce", "K7s1_scst", DEPLOYED) if x in SYS]:
            f.write(f"| {n}{' (deployed)' if n == DEPLOYED else ''} | {e[n]} | {e[n + '_labels']} |\n")
        f.write("\n")
print(f"\nqualitative examples ({len(EXS)}), ground truth vs generated:")
for e in EXS:
    print(f"\n  [{e['idx']}] {e['why']}\n    GT      : {e['GT'][:200]}\n    {DEPLOYED[:8]:8s}: {e[DEPLOYED][:200]}\n    labels GT {e['GT_labels']} | out {e[DEPLOYED + '_labels']}")

NPYS = [p for p in glob.glob(f"{INPUT_ROOT}/**/images_512.npy", recursive=True)]
if NPYS:
    imgs = np.load(NPYS[0], mmap_mode="r")
    show = EXS[:8]
    fig, ax = plt.subplots(2, 4, figsize=(20, 12))
    for a, e in zip(ax.ravel(), show):
        im = np.asarray(imgs[int(e["idx"])]).squeeze()
        a.imshow(im if im.ndim == 2 else im[..., 0], cmap="gray"); a.axis("off")
        txt = (f"{e['why']}\nGT: " + "\n".join(textwrap.wrap(e["GT"], 55)[:4]) + f"\nOUT: " + "\n".join(textwrap.wrap(e[DEPLOYED], 55)[:4]))
        a.set_title(txt, fontsize=8, loc="left")
    for a in ax.ravel()[len(show):]:
        a.axis("off")
    plt.tight_layout(); plt.savefig(f"{WORK}/eval_examples.png", dpi=110); plt.close()
    print("saved eval_examples.png")
else:
    print("images_512.npy not attached (IE643 Cleaning): eval_examples.png skipped, text examples saved")

# training curves of the deployed model's lineage (stage 1 -> KL-SCST -> F2) and the CE-only baseline
LINEAGE = [n for n in ("A0_ce", "A7s1_ce_feat", "K7s1_scst", "K10s1_fb2") if n in FOUND and os.path.exists(os.path.join(os.path.dirname(FOUND[n]), "curve.jsonl"))]
if LINEAGE:
    fig, ax = plt.subplots(1, 3, figsize=(17, 4.5))
    off = 0
    for n in LINEAGE:
        cv = [json.loads(l) for l in open(os.path.join(os.path.dirname(FOUND[n]), "curve.jsonl"))]
        scst = n.startswith("K") or any("val_ce5_f1" in c for c in cv)
        x = lambda c: c["step"] if scst else c["epoch"]
        if not scst:
            ax[0].plot([c["epoch"] for c in cv if "train_ce" in c], [c["train_ce"] for c in cv if "train_ce" in c], marker="o", label=f"{n} train CE")
            ax[0].plot([c["epoch"] for c in cv], [c["val_ce"] for c in cv], marker="s", ls="--", label=f"{n} val CE")
            ev = [c for c in cv if "val_ce14_f1" in c]
            ax[1].plot([c["epoch"] for c in ev], [c["val_ce14_f1"] for c in ev], marker="o", label=n)
        else:
            for k, ls in (("val_ce14_f1", "-"), ("val_ce5_f1", "--")):
                ev = [c for c in cv if k in c]
                ax[2].plot([c["step"] for c in ev], [c[k] for c in ev], ls=ls, marker=".", label=f"{n} {k[4:]}")
    for a, t, xl in zip(ax, ["stage 1: cross-entropy", "stage 1: val CheXbert-14 F1 (greedy)", "stage 2 (KL-SCST / F2): val CheXbert F1"], ["epoch", "epoch", "SCST step"]):
        a.set_title(t); a.set_xlabel(xl); a.grid(alpha=.3); a.legend(fontsize=7)
    plt.tight_layout(); plt.savefig(f"{WORK}/eval_training_lineage.png", dpi=130); plt.close()
    print("saved eval_training_lineage.png for", LINEAGE)
else:
    print("no curve.jsonl found for the lineage runs: training curves skipped (they are in the distill notebooks' training_curves.png)")
TAB.to_csv(f"{WORK}/eval_master_table_full.csv")
print(f"saved eval_key_finding_agreement.csv, eval_examples.csv/.md, eval_master_table_full.csv | total {(time.time() - T0) / 60:.1f} min\nEVAL EXTRAS DONE")
